# Prosody verifier for Shaer: main notebook

**Notebook version 1.1.1** · updated 2026-11-24 UTC. Section 2 prints the version of the code it downloaded and warns if this notebook copy is too old for it.

Research line on top of the reproduced Shaer baseline: a formal Arabic prosody verifier (meter, qafiya, radeef,
poem-level consistency) used to audit the baseline and to guide generation (idea 1: verifier-guided decoding).

* All code for this research lives in [`prosody-verifier/`](https://github.com/thebazshah/Shaer/tree/dev/prosody-verifier) on branch `dev`.
  Every cell is plain Python calling `prosody-verifier/pv_colab.py`, which reuses the baseline module `colab/shaer_colab.py`
  (unchanged) for the environment, progress bars, heartbeat and stall detection. Only the installation cell uses `!` commands.
* Everything this notebook writes goes to **`MyDrive/shaer-prosody/`**. Baseline results are only read, from
  `MyDrive/shaer-repro-yehia/runs/` where the baseline notebook put them.
* **Saving to Drive.** Every step works on the VM's own disk and uploads its results to `MyDrive/shaer-prosody/` through the
  Drive API, each file checked by MD5 (`☁ Drive: everything this step wrote is on Drive`): in the background about every minute,
  and once more when the step finishes or is stopped (a stop then halts *Run all*). A step first copies back from Drive what it
  resumes from, so a new session continues exactly where the last upload left it. The first step of a session asks once for
  Google's permission to use Drive. (The Drive mount alone uploads lazily: a re-run could resume from an older point.)
* Every long step shows progress and a heartbeat; a step that is silent and idle for `STALL_MINUTES` is stopped with a message,
  and re-running the cell resumes.

**Runtime.** M0, M1, M3 and all scoring need no GPU: pick a CPU runtime (cheapest). Generation (M4, M5) and training (M5) need a GPU; an A100 is fastest, an L4 works with smaller batches.

| Milestone | What | Runtime | Status |
|---|---|---|---|
| M0 | Headroom audit of the baseline poems and the human reference poems | CPU, minutes | in this notebook |
| M1 | The prosody verifier: rule tables, scansion, rhyme, poem verdicts, validation | CPU | in this notebook |
| M2 | Expert-checked gold set and sign-off of the rule tables | CPU | added later |
| M3 | Strict re-scoring of the baselines with the verifier | CPU, about 1 hour per system | in this notebook |
| M4 | Verifier-guided decoding (idea 1) vs best-of-N at equal compute | GPU, then CPU | in this notebook |
| M5 | Self-training (idea 2): rhyme letter in the prompt (2a), verifier-clean data (2b), rejection-sampling rounds (2c) | GPU (training, generation) and CPU | in this notebook |
| M6 | Quality study: LLM judge, copying and diversity, blinded human ratings | CPU (the judge is a paid API) | in this notebook |

## 1. Configuration, Google Drive, secrets

In [ ]:
import getpass
from google.colab import drive, userdata

NOTEBOOK_VERSION = "1.1.1"   # updated 2026-11-24 UTC; section 2 checks it against the downloaded code

# ---- edit these -------------------------------------------------------------------------------------------
REPO_URL = "https://github.com/thebazshah/Shaer.git"
BRANCH = "dev"
BASE_MODEL = "Navid-AI/Yehia-7B-preview"   # the paper's model; it selects the baseline folders (shaer-repro-yehia)
STALL_MINUTES = 15     # a step that is silent AND idle for this long is stopped, with instructions to resume
# -----------------------------------------------------------------------------------------------------------

drive.mount("/content/drive")

def secret(name, prompt=None):
    try:
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    return getpass.getpass(prompt) if prompt else ""

HF_TOKEN = secret("HF_TOKEN")            # optional for M0 (the released poems and the meter classifier are public)
GITHUB_TOKEN = secret("GITHUB_TOKEN")    # only needed if the fork is private
OPENROUTER_API_KEY = secret("OPENROUTER_API_KEY")   # only for the LLM judge (later milestones)
print("HF token set:", bool(HF_TOKEN), "| OpenRouter key set:", bool(OPENROUTER_API_KEY))

## 2. Get the code from GitHub (branch `dev`)

In [ ]:
import importlib, pathlib, subprocess, sys

REPO = pathlib.Path("/content/Shaer")
git_url = REPO_URL.replace("https://", f"https://x-access-token:{GITHUB_TOKEN}@") if GITHUB_TOKEN else REPO_URL
if (REPO / ".git").exists():
    subprocess.run(["git", "-C", str(REPO), "checkout", "-q", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "-q", "--ff-only", git_url, BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "-q", "--branch", BRANCH, git_url, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "remote", "set-url", "origin", REPO_URL], check=True)   # keep no token in .git/config

for p in (REPO / "colab", REPO / "prosody-verifier"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
import shaer_colab as sc
import pv_colab as pv
importlib.reload(sc); importlib.reload(pv)      # pick up the latest versions after a pull

cfg = sc.make_config(base_model=BASE_MODEL, stall_minutes=STALL_MINUTES, repo_url=REPO_URL, branch=BRANCH,
                     hf_token=HF_TOKEN, openrouter_key=OPENROUTER_API_KEY)
# cfg is the baseline's configuration: its folders (MyDrive/shaer-repro-yehia: the E3 adapter in sft_outputs/train/<RUN_NAME>,
# the generations in runs/e0..e3) are only read here. pv.make_config replaces it by a copy whose outputs all go to the sibling
# MyDrive/shaer-repro-yehia-prosody (run names prosody_*) and points the repository's sft/outputs and runs/ links there.
pvc = pv.make_config(cfg)            # this research writes to MyDrive/shaer-prosody (and the sibling above for training)
cfg = pvc.sc_cfg
pv.check_version(NOTEBOOK_VERSION)  # warns if this notebook copy is too old for the downloaded code
pv.runtime_info()

## 3. Environment (about 5 minutes on a fresh runtime)

The same isolated environment as the baseline notebook (`/content/venv`, the repository's pinned versions). Safe to run again:
when everything is installed it finishes in seconds.

In [ ]:
!pip -q install uv
!uv venv --python 3.12 --allow-existing /content/venv
!uv pip install --python /content/venv/bin/python torch==2.3.1 --index-url https://download.pytorch.org/whl/cu121
!uv pip install --python /content/venv/bin/python -r /content/Shaer/colab/requirements-colab.txt
!uv pip install --python /content/venv/bin/python pytest
!uv pip check --python /content/venv/bin/python

## 4. M0 - headroom audit

Where do the baseline poems fail, and how far are they from the human poems on the same prompts?

* **Systems:** the human reference poems of the 3,481 test prompts, the released Shaer poems (E0), and every complete baseline
  run found on Drive (E1 released adapter, E2 untuned base model, ...). Runs with several samples per prompt are audited on the
  first sample (the paper's protocol).
* **Count:** exact / too short / too long / odd final hemistich, by requested length.
* **Meter:** the repository's BiLSTM classifier per bayt: share of bayts on the requested meter, poems with every bayt on it,
  poems whose bayts get more than one meter, error rate by bayt position and by meter. Scored with a batched wrapper of the
  repository's own scorer (identical results, checked on every run; much faster on a CPU).
* **Rhyme:** a written-form **heuristic** of the rawi (rhyme letter) at the end of each bayt. It is *not* the prosody verifier;
  read it next to the reference column, which shows how the same heuristic behaves on human poems.
* **Diacritics:** how much of the text carries harakat, which tells the verifier how much vowelisation it must infer.

Outputs: `MyDrive/shaer-prosody/m0/` (per-poem rows, per-system summaries, `m0_report.md`, `m0_summary.json`).

In [ ]:
pv.run_m0(pvc)

In [ ]:
pv.show_m0(pvc)

## 5. M1 - the prosody verifier

The verifier checks a poem against a requested meter and form, the way a prosodist would:

* **Rule tables** (`prosody/feet.py`, `prosody/meters.py`): the 8 base feet, every zihāf and ʿilla as an operation, the 16
  meters with their forms and ʿarūḍ/ḍarb tables. Every pattern is derived, never typed by hand.
* **Prosodic writing** (`prosody/arudi.py`): what is pronounced, not what is written (waṣl, sun letters, shadda, tanwīn,
  hidden long vowels, lengthened final vowels...). Text with few or no diacritics allows several readings; all of them are
  searched, and a pronunciation prior learned from the **train** split (`data/lexicon_train.json.gz`) says which readings
  are natural, so a line cannot be forced into a meter through an implausible reading.
* **Scansion** (`prosody/scan.py`) of every hemistich against every meter, with the nearest pattern and the broken foot
  when a line fails.
* **Rhyme** (`prosody/qafiya.py`): rawī, its vowel (iqwāʾ / iṣrāf), ridf, taʾsīs, waṣl, khurūj, muṭlaqa/muqayyada, īṭāʾ,
  and an Urdu-style refrain (radīf) when the poem has one.
* **Poem verdicts** (`prosody/verify.py`): allowed ʿarūḍ/ḍarb pairs, the same ḍarb (and ʿarūḍ) in every bayt, taṣrīʿ in the
  opening bayt, tadwīr, length.

Validation (CPU only): the unit tests; human poems of the **eval** split (tuning) and **test** split (report) as positives;
systematically broken copies as negatives; planted rhyme breaks; and bayt-by-bayt agreement with the BiLSTM classifier on
the released Shaer poems and on the human poems (needs M0). Everything is written to `MyDrive/shaer-prosody/m1/`.
`limit=0` runs a whole split (about 40 minutes per split on a 2-core CPU runtime).

In [ ]:
pv.m1_tests(pvc)                   # rule tables, prosodic writing, scansion, rhyme, poem verdicts

In [ ]:
LEXICON = pv.m1_lexicon(pvc)       # the shipped lexicon; pv.m1_lexicon(pvc, rebuild=True) rebuilds it from the train split

In [ ]:
pv.m1_validate(pvc, "eval", limit=1000)    # tuning split
pv.m1_validate(pvc, "test", limit=1000)    # report split (limit=0: all 3,481 poems)

In [ ]:
pv.m1_compare_bilstm(pvc, limit=1000)     # verifier vs the BiLSTM on released Shaer poems and on human poems

In [ ]:
pv.m1_report(pvc)

**Try it on any poem.** `base_meter` and `form` use the Shaer labels (الطويل، الكامل ... / تام، مجزوء، مخلع، أحذ).
`ignore_diacritics=True` judges only the letters, so wrong written vowels do not count.

In [ ]:
poem = """قِفَا نَبْكِ مِنْ ذِكْرَى حَبِيبٍ وَمَنْزِلِ
بِسِقْطِ اللِّوَى بَيْنَ الدَّخُولِ فَحَوْمَلِ
فَتُوضِحَ فَالْمِقْرَاةِ لَمْ يَعْفُ رَسْمُهَا
لِمَا نَسَجَتْهَا مِنْ جَنُوبٍ وَشَمْأَلِ"""
report = pv.verify_poem(poem, "الطويل", requested_lines=4)

## 6. M3 - strict re-scoring of the baselines

Every system is re-scored with the verifier: the human reference poems, the released Shaer poems (E0), and every complete
baseline run on Drive (E1-E3). Shaer's own metrics (computed by the repository's `summarize_structural.py`, unchanged) are
shown next to the strict ones:

* **strictly valid poem**: every bayt in the requested meter and form, the ḍarb (and ʿarūḍ) form kept, the rhyme check
  passes, the exact number of hemistichs, no odd hemistich;
* meter, rhyme and length separately; bayts on meter; bayts with **no reading at all** in the requested meter (independent
  of the margin); the BiLSTM's per-bayt verdict on the same bayts;
* two reading modes: *letters* (written vowels ignored; the conservative meter verdict) and *binding* (written vowels
  count); margins 1.0, 1.5 (default), 3.0 and none;
* 95% bootstrap intervals over prompts, paired differences to the human poems, results by length, meter and bayt
  position, rhyme flaws by kind, pass@k when a run has several samples per prompt;
* two controls built from the human poems: each poem judged against the wrong meter, and poems whose halves come from
  two poems with different rhymes;
* integrity checks that must all be 0 (line counts against Shaer's parser, BiLSTM bayt alignment, margin recomputation,
  exact length against the official metric).

CPU runtime. Needs M0 (section 4). Each finished poem is written to `MyDrive/shaer-prosody/m3/` at once, so re-running a
cell resumes. The validation of this step is in `prosody-verifier/validation/M3_report.md`.

In [ ]:
pv.m3_rescore(pvc)        # all systems found on Drive; pv.m3_rescore(pvc, limit=200) for a quick look

In [ ]:
pv.m3_report(pvc)

## 7. M4 - verifier-guided decoding against best-of-N, at equal compute

Two ways of spending more samples on one prompt, both with the released Shaer adapter on Yehia-7B-preview and the paper's
decoding (temperature 0.55, top-p 0.90, repetition penalty 1.08):

* **I0 / I1, best-of-N:** sample N whole poems; keep the first strictly valid one (letters mode, margin 1.5), else the one
  with the fewest defects. I0 is N = 1, the paper's setting.
* **I2, hemistich search:** write the poem one hemistich at a time; for each, sample k candidates (up to R rounds) and keep
  the first that adds no new defect to the poem so far (meter, ḍarb, rhyme with the rawī fixed by the first bayt, no
  repeated rhyme word); else the one adding the fewest. Exactly the requested number of hemistichs is written.

Both keep the model's own first acceptable sample, so the verifier filters but never picks the "most metrical" line.
The budget is every generated token, rejected candidates included. Each search run is compared with best-of-N at the
smallest N that used at least as many tokens, with paired bootstrap intervals and an exact McNemar test. The selecting
verifier is also the scorer, so the report also shows the BiLSTM metric of the paper, binding mode and no margin, plus
repetition, diversity and copying from the training poems.

**Order** (each step resumes; every finished poem is on Drive under `MyDrive/shaer-prosody/m4/`):

1. `m4_selfcheck`: a few minutes on the GPU; it must say PASSED before the long runs.
2. `m4_dev`: the development sweep on 400 seeded eval prompts, on the GPU. It makes one run of 16 samples per prompt
   (the long one, about 2 h on an A100) and five search runs (minutes to tens of minutes each). Each run loads the model
   again (a few minutes).
3. `m4_score` and `m4_report` on `"eval"`: a CPU runtime is enough. The report prints the test configuration chosen by the
   rule fixed in advance: the search run with the highest strict validity among those within the budget of best-of-4.
4. `m4_test_from_choice`: the chosen search run and the matching best-of-N on the 3,481 test prompts (GPU). Best-of-N
   takes N times one sample's time, about 1.5 h per sample on an A100.
5. `m4_score` and `m4_report` on `"test"`.

`m4_check_sampling` (optional, about 20 min on the GPU) checks that the sample mode reproduces the baseline's E1
decoding on 300 test prompts. An L4 works too, with smaller batches (about 3-5 times slower). The validation is in
`prosody-verifier/validation/M4_report.md`.

In [ ]:
pv.m4_selfcheck(pvc)          # GPU: shared cache vs full forward pass; stepwise vs one-shot greedy decoding

In [ ]:
pv.m4_dev(pvc)                # GPU: development sweep on the eval split; pv.m4_dev(pvc, only=("i2_k4_r1",)) runs one

In [ ]:
pv.m4_score(pvc, "eval")      # CPU is enough: the repository's scorer and the M3 verifier on every run
pv.m4_report(pvc, "eval")

In [ ]:
pv.m4_test_from_choice(pvc)   # GPU: the pre-registered test runs

In [ ]:
pv.m4_score(pvc, "test")
pv.m4_report(pvc, "test")

In [ ]:
pv.m4_check_sampling(pvc)     # optional, GPU: the sample mode against the baseline's E1 poems

## 8. M5 - self-training on verifier-approved poems (idea 2)

The released model is never told which rhyme to keep, and rhyme is where it fails most (38% of its test poems pass the
rhyme check, 93% of the human ones). M5 trains new adapters with the baseline's recipe (QLoRA on Yehia-7B-preview,
`sft/train_sft.py` unchanged) on new data:

* **2a**: every prompt names the rhyme letter of its poem (`حرف الروي: ...`, the verifier's rawī), so the model learns to
  follow a requested rhyme; at test time it is asked for the rawī of the human poem of the prompt.
* **2b**: 2a, trained only on the poems the verifier accepts (meter with no margin, and rhyme).
* **2c**: rejection-sampling rounds: the 2a/2b model writes poems for train prompts with M4's hemistich search, a quality
  gate keeps the strictly valid ones that copy nothing from the corpus, and one epoch of training on them (mixed with as
  many original poems) continues the adapter. Up to two rounds.

Each variant is judged by **one ordinary sample per test prompt** (no search, no selection) against the baseline's E3
(the same recipe without the rhyme field) and E1 (the released adapter), and optionally the authors' best GRPO adapter.

**Where things go.** Checkpoints: `MyDrive/shaer-repro-yehia-prosody/sft_outputs/train/prosody_m5_<variant>/`, beside the
baseline's folder, which is never written. A resume checkpoint is written about every 10 minutes. Data, labels and results:
`MyDrive/shaer-prosody/m5/`. Every step resumes.

**Cost.** One training run of 2a or 2b costs about as much as E3 (two epochs of the train split: roughly 12-20 h on an
A100). A 2c round: the search on 4,000 train prompts (1-2 h) and one short epoch (2-3 h). Each test generation: about
1.5 h. Run 2a first; go on to 2b and 2c only if 2a's report shows a gain. The validation is in
`prosody-verifier/validation/M5_report.md`.

In [ ]:
pv.m5_relabel(pvc)                 # CPU: the verifier's labels of the corpus (rawī, meter, rhyme); resumable

In [ ]:
pv.m5_build(pvc, "2a")             # CPU: the training set with the rhyme letter in every prompt

In [ ]:
pv.m5_train(pvc, "2a")             # GPU: training with live progress; re-run the cell to resume

In [ ]:
pv.m5_generate(pvc, "2a")          # GPU: one sample per test prompt with the best 2a checkpoint
pv.m5_import_baselines(pvc)        # the baseline's E1 / E3 test poems (read only), for the comparison

In [ ]:
pv.m5_score(pvc)                   # CPU
pv.m5_report(pvc)

**Only if 2a gains** (strict validity and rhyme above E3, the quality section no worse): 2b, then the 2c rounds.
`pv.m5_generate(pvc, "grpo_v2")` adds the authors' best GRPO adapter to the comparison (GPU, about 1.5 h).

In [ ]:
pv.m5_build(pvc, "2b")
pv.m5_train(pvc, "2b")
pv.m5_generate(pvc, "2b")

In [ ]:
pv.m5_selftrain(pvc, round_no=1, base="2a")    # or base="2b" if it did better; GPU
pv.m5_generate(pvc, "2c_r1")

In [ ]:
pv.m5_score(pvc)
pv.m5_report(pvc)

## 9. M6 - quality of the final systems

Optimising against the verifier can make poems valid and worse. M6 measures what the verifier cannot, on the final
systems (the M4 test conditions, the M5 models and the human poems of the same prompts):

* the paper's **LLM judge** (description adherence, meaning, fluency, coherence, poeticness; 1-5), on the same prompts for
  every system. It is a **paid API** (OpenRouter, one poem = five calls; the Colab secret `OPENROUTER_API_KEY`): start
  with `dry_run=True`, then a few rows;
* **copying and diversity** (lines copied from the training poems, repeated lines, distinct n-grams, distinct rhyme words);
* a **blinded human study**: a rating sheet (CSV for Google Sheets) with the systems' poems shuffled under item codes, and
  its key kept apart; filled copies go to `MyDrive/shaer-prosody/m6/human/filled/`.

Summarise M4's test runs (`pv.m4_report(pvc, "test")`) and M5's first. A CPU runtime is enough. The validation is in
`prosody-verifier/validation/M6_report.md`.

In [ ]:
pv.m6_collect(pvc)                  # one poem per test prompt for every final system, and the human poems

In [ ]:
pv.m6_judge(pvc, dry_run=True)      # the plan and one exact prompt; no API call, no cost

In [ ]:
pv.m6_judge(pvc, limit_rows=300, max_cost_usd=10)   # paid: the same 300 prompts for every system; resumable

In [ ]:
SHEET = pv.m6_human_sheet(pvc, names=("human", "e3_retrained", "m5_2a"), prompts=40)   # pick the systems to rate
# give each rater a copy; put the filled CSVs in MyDrive/shaer-prosody/m6/human/filled/, then:
# pv.m6_human_score(pvc)

In [ ]:
pv.m6_report(pvc)

## Next milestones

M2 (the expert gold set) is added when a prosody expert is available. Pull the latest `dev` (section 2) before running them.
If you save this notebook from Colab, save it under a different name (or reload it from GitHub first), so a stale copy does
not overwrite newer cells.